# CSE 152A: Discussion Week 9: Neural Networks And PyTorch Continued
## Announcements
- HW4 is released
- We recommend you use Google Colab for the free GPU services (it will likely take a long time just to run the cells on CPU). It's important to start early just in case you run into issues (run out of compute units, etc).

## Agenda
- Backpropagation and Autograd
- Connecting concepts NN concepts in lecture to PyTorch code

## **Set-up**
### First, lets import PyTorch and some useful functions/objects

In [ ]:
import torch # Main torch package
from torch import nn # Importing specifically the nn class of the torch library, this will make our code more consise later on
from torch.utils.data import DataLoader # Importing the DataLoader class
from torchvision import datasets # Used for importing built-in datasets
from torchvision.transforms import ToTensor # Used to transform data to tensors (the main object in PyTorch)
import matplotlib.pyplot as plt # Plotting
import numpy as np
from tqdm import tqdm # Progress bar

if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"

print(f"The device currently available is: {device}")
!nvidia-smi # This will show information about your GPU if there is one available

# Backpropagation

## Consider a single layer perceptron:
$$f(x) = Wx + b$$
where $x \in \mathbb{R}^2$, $W \in \mathbb{R}^{1 \times 2}$, $b \in \mathbb{R}$.

## **Question 1**: Backpropagation Algorithm
If $x = \begin{bmatrix} 2 \\ -1 \end{bmatrix}$, $W = \begin{bmatrix} -4, 2 \end{bmatrix}$, $b = 3$ and the ground truth label for this datapoint $y = -5$, using mean squared error (MSE) as our loss function, compute the gradient with respect to each parameter $w_1$, $w_2$, and $b$. MSE is defined as:
$$ \frac{1}{n} \sum_{x_i} (y - f(x_i))^2$$

After computing it manually, can we show the same thing in code using PyTorch's Autograd?

In [ ]:
# Initialize the data and parameters
x_1 = torch.tensor([])
x_2 = torch.tensor([])
y = torch.tensor([])
# We want to compute the gradients of the loss function w.r.t. each of the parameters
# So, need to turn on gradient tracking
w_1 = torch.tensor([], requires_grad=True)
w_2 = torch.tensor([], requires_grad=True)
b = torch.tensor([], requires_grad=True)

# Compute f(x) with the given data and parameter initializations
f =
print(f"Output of network: {f.item()}")

# Now, compute the loss
loss =
print(f"Loss: {loss.item()}")

Great! Looks like forward propagation matches our results. Now, let's check gradients computed from backwards propagation.

In [ ]:
# Call .backward()
loss.backward()

# Print the gradients w.r.t. w_1, w_2, b
print(f"Gradient w.r.t. w_1: {w_1.grad}")
print(f"Gradient w.r.t. w_2: {w_2.grad}")
print(f"Gradient w.r.t. b: {b.grad}")

As you can see, it matches our manual computation exactly. You could further break this up into smaller pieces just like we did originally and everything will still be the same (i.e. $i = w_1 \times x_1$, $j = w_2 \times x_2$, etc).

## **Question 2** Gradient Descent and Update Rule
Now that we know the direction in which we need to adjust the weights $w_1$, $w_2$, and $b$, write the update rule. Choose the step size to be $0.1$.

<details>
  <summary> Solution </summary>
The update rule for any weight $w$ is given by:
$$ w' = w - \eta \frac{\partial C}{\partial w} $$
where $w'$ is the weight at the next iteration, $w$ is the weight at the current iteration, $\eta$ is the step size, and $C$ is the cost (loss) function.

So, the update rules are as follows:
$$ w'_1 = w_1 - (0.1)*\frac{\partial C}{\partial w_1} = -4 - (0.1)(-8) = -3.2 $$
$$ w'_2 = w_2 - (0.1)*\frac{\partial C}{\partial w_2} = 2 - (0.1)(4) = 1.6$$
$$ b' = b - (0.1)*\frac{\partial C}{\partial b} = 3 - (0.1)(-4) = 3.4 $$

Let's just do this in code to show the optima it heads towards.
</details>

In [ ]:
# Reinitialize data
x_1 = torch.tensor([2.])
x_2 = torch.tensor([-1.])
y = torch.tensor([-5.])
w_1 = torch.tensor([-4.], requires_grad=True)
w_2 = torch.tensor([2.], requires_grad=True)
b = torch.tensor([3.], requires_grad=True)


# Learning rate and iterations
lr = 0.1
num_epochs = 100

# Gradient descent loop
for epoch in range(num_epochs):
    # Forward pass
    y_pred = (w_1 * x_1) + (w_2 * x_2) + b
    loss = torch.square((y - y_pred))

    # Backward pass
    loss.backward()

    # Gradient descent update
    with torch.no_grad():
        w_1 -= lr * w_1.grad
        w_2 -= lr * w_2.grad
        b -= lr * b.grad

    # Zero gradients
    w_1.grad.zero_()
    w_2.grad.zero_()
    b.grad.zero_()

    # Print loss every 10 epochs
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

# Final optimal values
print("\nOptimal Parameters:")
print(f'W: {W.detach().numpy()}')
print(f'b: {b.item()}')
# Show result of network
print(f"Forward Propagation with optimized parameters: {((w_1 * x_1) + (w_2 * x_2) + b).item()}")
print(f"Ground Truth: {y.item()}")

## **Connecting Conceptual Model to Code: Initializing the Dataset**
### Let's work with the MNIST dataset: [MNIST](https://en.wikipedia.org/wiki/MNIST_database). This dataset has:
- 10 classes (different random objects)
- $28 \times 28$ grayscale images (1 channel)

In [ ]:
# Download datasets
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

testing_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

# Create data loaders
batch_size = 128
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(testing_data, batch_size=batch_size)

for X,y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")

    plt.imshow(X[1,0,:,:], cmap="gray") # Show an image from our dataset
    print(f"This image is of class {y[1]}")
    break

### Remember, the standard order of dimensions in PyTorch (for images) is of shape: (batch_size, channels, height, width).

## **Connecting Conceptual Model to Code: Training and Evaluating**

Just like last time, we have a training loop and evaluation function. Nothing has changed here, so let's initialize it first this time.

[Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)

[CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

In [ ]:
def train(dataloader, model, loss_fn, optimizer, device):
    """
    This function will perform the optimization/training of the network
    """
    size = len(dataloader.dataset) # Grab the total number of images
    model.train() # Set the model to training mode (parameters can be updated and gradients are calculated)
    optimizer.zero_grad() # Just make sure that the gradients are zero at the beginning
    loss_history = []

    # Now, we loop over our training data, pass in a batch, calculate the loss,
    # calculate gradients, and update the our weights through the optimizer
    for batch, (X, y) in enumerate(dataloader):

        # First, we need to move the data to the same device as the model
        # Ideally, everything is done on GPU
        X, y = X.to(device), y.to(device)

        # Then, we can just pass the data through the model (forward)
        # and calculate the loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Now, we can calculate the gradients through backpropagation
        loss.backward()

        # Take a step along the direction of the gradient (minimize)
        optimizer.step()

        # Zero out the gradients
        optimizer.zero_grad()

        # Logging
        loss, current = loss.item(), (batch+1)*len(X)
        if batch % 100 == 0:
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        loss_history.append(loss)

    return loss_history

def test(dataloader, model, loss_fn, device):
    """
    This function will test/evaluate our network after training
    """
    size = len(dataloader.dataset) # Grab the total number of images
    num_batches = len(dataloader) # Grab the total number of batches
    model.eval() # Set the network to eval mode
    test_loss, correct = 0.0, 0.0
    with torch.no_grad(): # Ensure we do not calculate gradients
        for X, y in dataloader:

            # Send data to same device as model
            X, y = X.to(device), y.to(device)

            # Pass the data through the model
            pred = model(X)

            # Compute the loss on the test set
            test_loss += loss_fn(pred, y).item()

            # Compute the number of correct predictions
            correct += (pred.argmax(dim=1) == y).type(torch.float).sum().item()

    # Get the average test loss
    test_loss /= num_batches

    # Compute the accuracy
    acc = correct / size

    print(f"\nTest Error: \n Accuracy: {(100*acc):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## **Part 3: Defining the network**

### Last time, we looked a small convolutional neural network that obtained poor accuracy. Here, we will define two different, slightly larger models:
1. Multi-layer Perceptron
2. Convolutional Neural Network

### We utilized the standard PyTorch classes to define each of these last time (linked below again for your convenience).

[Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

[Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)

[ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

[Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)

### **In HW4, you should utilize these classes, they make things very easy.** However, it abstracts away the mathematical operations that our model is performing. For conceptual understanding this time, let's avoid using the pre-defined classes and implement the mathematical operations ourselves.

---------------------------------------------------------------
### *Network 1: Multi-layer Perceptron (MLP)*
This network will only consist of fully-connected layers (known as Linear layers in PyTorch). We will not include any convolutional layers. This network will have the following:
- Fully-connected layer with 512 output features (with bias)
- ReLU activation layer
- Fully-connected layer with 256 output features (with bias)
- ReLU activation layer
- Fully-connected layer with 10 output features (with bias)

For simplicity's sake, let's use the Torch implementation for ReLU (the gradient is handled differently at `x=0` for `torch.maximum()` and `torch.clamp`). Within the constructor, we need to initialize the parameters and then in the forward, we need to perform the mathematical operations.

## **Question 3**: Neural networks are essentially complex functions. How do we express the network described as a function?

## Remember: a fully connected layer simply just describes an affine function, where A is a weight matrix and b is a bias vector.
$$g_{\text{Linear}}(x) = Ax+b$$

## Now, write the network above as a function $f(x)$:

<details>
  <summary> Solution </summary>
Given some weight matrices $W_1$, $W_2$, $W_3$ and some bias vectors $b_1$, $b_2$, $b_3$, we can just apply the matrix-vector products and vector additions in order:
$$f(x) = W_3\text{relu}(W_2(\text{relu}(W_1x+b_1)+b_2)+b_3$$

Now, let's try to define it in the code.
</details>

In [ ]:
class MLP(nn.Module):
    def init_weights(self, in_features, out_features):
        """
        Initializes weight tensor of shape (out_features, in_features)
        with uniform initialization U(-sqrt(k), sqrt(k)) as described
        in nn.Linear().
        https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
        """
        # Compute sqrt(k)
        k = (1.0 / in_features)
        sqrt_k = torch.sqrt(torch.tensor(k)).item()
        # Initialize the weight matrix
        weight = torch.empty((out_features, in_features)).uniform_(-sqrt_k, sqrt_k)
        # Turn gradient tracking on and add to our parameter list
        return nn.Parameter(weight)

    def init_bias(self, in_features, out_features):
        """
        Initializes bias tensor of shape (out_features,)
        with uniform initialization U(-sqrt(k), sqrt(k)) as described
        in nn.Linear().
        https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
        """
        # Compute sqrt(k)
        k = (1.0 / in_features)
        sqrt_k = torch.sqrt(torch.tensor(k)).item()
        # Initialize the weight matrix
        bias = torch.empty((out_features,)).uniform_(-sqrt_k, sqrt_k)
        # Turn gradient tracking on and add to our parameter list
        return nn.Parameter(bias)

    def __init__(self):
        """
        Multi-layer Perceptron as defined above without the use of the nn.Linear() class.
        Within the constructor, we need to define the parameters we will use.
        """
        super().__init__()
        # Now, let's initialize our weights using the helper functions above
        self.fc1_weights = self.init_weights()
        self.fc1_bias = self.init_bias()

        self.fc2_weights = self.init_weights()
        self.fc2_bias = self.init_bias()

        self.fc3_weights = self.init_weights()
        self.fc3_bias = self.init_bias()

        self.relu = nn.ReLU() # Nonlinear Activation
        self.flatten = nn.Flatten() # Used for flattening the tensor for fc layers

    def forward(self, x):
        """
        Now, we need to perform the mathematical operations that our
        neural network describes using the data x and parameters
        """
        # First, need to flatten our images
        # (batch_size, channels, height, width) -> (batch_size, channels*height*width)
        x = self.flatten(x)

        # First fc layer
        # self.fc1_weights -> (out_features1, in_features)
        # x -> (batch_size, in_features)
        # want: (batch_size, out_features1)
        out1 =
        out1 =

        # Second fc layer
        out2 =
        out2 =

        # Third fc layer
        logits =
        return logits

In [ ]:
# Now we can instantiate a network:
model = MLP() # Create/Initialize the model
model.to(device) # Send the model to the device we've chosen (either GPU or CPU depending on what's available)
loss_fn = nn.CrossEntropyLoss() # Initialize our loss function
optimizer = torch.optim.Adam(model.parameters()) # Initialize our optimizer -- we pass in the parameters that we want to optimize

# Now, we can just call the functions we've written
# Training
train_loss = train(train_dataloader, model, loss_fn, optimizer, device)

# Testing
test(test_dataloader, model, loss_fn, device)

# Now, we can plot our training loss
plt.plot(range(len(train_loss)), train_loss)
plt.xlabel("Batch Iteration")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

We've now shown the operations of a fully connected layer. Another important thing to know about your network is its size (how many parameters)

## **Question 4**: Can you calculate how many parameters in total this MLP has?
Remember: the number of parameters is equal to the number of weights (including biases) that need to be optimized.

<details>
  <summary> Solution </summary>
Our network has three FC layers with the following:
- FC1: 28*28 in_features, 512 out_features (with bias)
- FC2: 512 in_features, 256 out_features (with bias)
- FC3: 256 in_features, 10 out_features (with bias)

So, we should just count the values in each weight matrix and bias vector:
- FC1: $28 \times 28 \times 512 + 512 = 401,408 + 512 = 401,920$
- FC2: $512 \times 256 + 256 = 131,072 + 256 = 131,328$
- FC3: $256 \times 10 + 10 = 2,560 + 10 = 2,570$

Adding them all together:

$FC1 + FC2 + FC3 = 401,920 + 131,328 + 2,570 = 535,818$ total parameters

Note: ReLU does **NOT** have any learnable parameters.

Let's verify this in our code..

</details>

In [ ]:
print("Parameters")
total_params = 0
for name, param in model.named_parameters(): # Print out all of our parameters
    params = torch.prod(torch.tensor(param.shape)).item()
    total_params += params
    print(f"Weight: {name} --> {param.shape} ({params} parameters)")
print("--------------------")
print(f"Computed: {total_params}")
print(f"Expected: 535818")

### *Network 2: Convolutional Neural Network (CNN)*
This network will now use convolutional layers (known as Conv2d layers in PyTorch). This network will have the following:
- Convolutional layer with $3 \times 3$ kernels, 16 output channels, stride of 1, using zero padding (padding=1) (with bias)
- Max-pooling layer with $2 \times 2$ kernels, stride of 2
- ReLU activation layer
- Convolutional layer with $3 \times 3$ kernels, 32 output channels, stride of 1, using zero padding (padding=1) (with bias)
- Max-pooling layer with $2 \times 2$ kernels, stride of 2
- ReLU activation layer
- Fully-connected layer with 32 output features (with bias)
- ReLU activation layer
- Fully-connected layer with 10 output features (with bias)

Writing the operations for this will be a bit more involved (would need to implement convolution from scratch, etc). So, let's use most of the premade libraries except for `nn.Conv2d()`, we will use `nn.functional.conv2d()` instead to be more explicit about the kernel sizes. Again, we will not use `nn.Linear()`.

## **Question 5: Define the convolutional kernel parameters and implement the forward propagation without using `nn.Linear()`, `nn.Conv2d()`.**

In [ ]:
class CNN(nn.Module):
    def init_weights_conv(self, in_channels, height, width, out_channels):
        """
        Initializes weight tensor of shape (out_features, in_features)
        with uniform initialization U(-sqrt(k), sqrt(k)) as described
        in nn.Conv2d().
        https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
        """
        # Compute sqrt(k)
        k = (1.0 / (in_channels * height * width))
        sqrt_k = torch.sqrt(torch.tensor(k)).item()
        # Initialize the weight matrix
        weight = torch.empty((out_channels, in_channels, height, width)).uniform_(-sqrt_k, sqrt_k)
        # Turn gradient tracking on and add to our parameter list
        return nn.Parameter(weight)

    def init_bias_conv(self, in_channels, height, width, out_channels):
        """
        Initializes bias tensor of shape (out_features,)
        with uniform initialization U(-sqrt(k), sqrt(k)) as described
        in nn.Conv2d.
        https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
        """
        # Compute sqrt(k)
        k = (1.0 / (in_channels * height * width))
        sqrt_k = torch.sqrt(torch.tensor(k)).item()
        # Initialize the weight matrix
        bias = torch.empty((out_channels,)).uniform_(-sqrt_k, sqrt_k)
        # Turn gradient tracking on and add to our parameter list
        return nn.Parameter(bias)

    def init_weights_fc(self, in_features, out_features):
        """
        Initializes weight tensor of shape (out_features, in_features)
        with uniform initialization U(-sqrt(k), sqrt(k)) as described
        in nn.Linear().
        https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
        """
        # Compute sqrt(k)
        k = (1.0 / in_features)
        sqrt_k = torch.sqrt(torch.tensor(k)).item()
        # Initialize the weight matrix
        weight = torch.empty((out_features, in_features)).uniform_(-sqrt_k, sqrt_k)
        # Turn gradient tracking on and add to our parameter list
        return nn.Parameter(weight)

    def init_bias_fc(self, in_features, out_features):
        """
        Initializes bias tensor of shape (out_features,)
        with uniform initialization U(-sqrt(k), sqrt(k)) as described
        in nn.Linear().
        https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
        """
        # Compute sqrt(k)
        k = (1.0 / in_features)
        sqrt_k = torch.sqrt(torch.tensor(k)).item()
        # Initialize the weight matrix
        bias = torch.empty((out_features,)).uniform_(-sqrt_k, sqrt_k)
        # Turn gradient tracking on and add to our parameter list
        return nn.Parameter(bias)

    def __init__(self):
        """
        Convolutional Neural Network (CNN) as defined above.
        Within the constructor, we need to define the parameters we will use.
        """
        super().__init__()
        # Now, let's initialize our weights using the helper functions above
        # Convolutional layers need: (in_channels, height, width, out_channels)
        self.conv1_weights = self.init_weights_conv()
        self.conv1_bias = self.init_bias_conv()

        self.conv2_weights = self.init_weights_conv()
        self.conv2_bias = self.init_bias_conv()

        # Linear layers need: (in_features, out_features)
        self.fc1_weights = self.init_weights_fc()
        self.fc1_bias = self.init_bias_fc()

        self.fc2_weights = self.init_weights_fc()
        self.fc2_bias = self.init_bias_fc()

        self.relu = nn.ReLU() # Nonlinear Activation
        self.pool = nn.MaxPool2d(2, stride=2) # Max Pooling layer
        self.flatten = nn.Flatten() # Used for flattening the tensor for fc layers

    def forward(self, x):
        """
        Now, we need to perform the mathematical operations that our
        neural network describes using the data x and parameters
        """
        # Pass through convolutional layers using nn.functional.conv2d()
        out1 = nn.functional.conv2d(
            x,
            self.conv1_weights,
            self.conv1_bias,
            stride=1,
            padding=1
        )
        out1 = self.pool(out1)
        out1 = self.relu(out1)

        out2 = nn.functional.conv2d(
            out1,
            self.conv2_weights,
            self.conv2_bias,
            stride=1,
            padding=1
        )
        out2 = self.pool(out2)
        out2 = self.relu(out2)

        # Flatten for fc layers
        flat = self.flatten(out2)

        # Pass through linear layers
        out3 = flat @ self.fc1_weights.T + self.fc1_bias
        out3 = self.relu(out3)

        # Second fc layer
        logits = out3 @ self.fc2_weights.T + self.fc2_bias
        return logits

In [ ]:
# Now we can instantiate a network:
model = CNN() # Create/Initialize the model
model.to(device) # Send the model to the device we've chosen (either GPU or CPU depending on what's available)
loss_fn = nn.CrossEntropyLoss() # Initialize our loss function
optimizer = torch.optim.Adam(model.parameters()) # Initialize our optimizer -- we pass in the parameters that we want to optimize

# Now, we can just call the functions we've written
# Training
train_loss = train(train_dataloader, model, loss_fn, optimizer, device)

# Testing
test(test_dataloader, model, loss_fn, device)

# Now, we can plot our training loss
plt.plot(range(len(train_loss)), train_loss)
plt.xlabel("Batch Iteration")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

## **Question 6**: Calculate the number of parameters this CNN has.

<details>
  <summary> Solution </summary>
Our network has two convolutional layers and two FC layers:

- Conv1:
    - Each kernel is $3 \times 3$, input data has 1 channel.
    - $3 \times 3 \times 1$ weights per kernel
    - There are 16 output channels, so we need 16 kernels
    - $3 \times 3 \times 1 \times 16 = 144$ weights in total
    - There is 1 bias for each output channel --> 16 biases
    - $144 + 16 = 160$ total parameters
- Conv2:
    - Input to this layer has 16 channels
    - Kernel is $3 \times 3$, with 32 output channels
    - $3 \times 3 \times 16 \times 32 = 4,608$ weights
    - $4,608 + 32 = 4,640$ total parameters
- FC1
    - We started with $28 \times 28 \times 1$ images.
    - Each max pooling layer halved the spatial resolution (2 max pooling layers) --> $\frac{28}/{4} = 7$
    - We have $7 \times 7$ images. There are 32 output channels from the last convolutional layer
    - There are 32 output features in this layer
    - So, the weight matrix has $7 \times 7 \times 32 \times 32 = 50,176$ weights
    - 32 biases
    - $50,176 + 32 = 50,208$ total parameters
- FC2
    - 32 input features, 10 output features
    - $32 \times 10 = 320$ weights plus 10 biases
    - $320 + 10 = 330$ weights

Adding them all together:

**$Conv1 + Conv2 + FC1 + FC2 = 160 + 4,640 + 50,200 + 330 = 55,338$ total parameters**

Note: ReLU and Max Pooling does **NOT** have any learnable parameters.

Let's verify this in our code..

</details>


In [ ]:
print("Parameters")
total_params = 0
for name, param in model.named_parameters(): # Print out all of our parameters
    params = torch.prod(torch.tensor(param.shape)).item()
    total_params += params
    print(f"Weight: {name} --> {param.shape} ({params} parameters)")
print("--------------------")
print(f"Computed: {total_params}")
print(f"Expected: 55338")
print(f"Fully-connected Network: 535818")
print(f"Ratio: {535818 / 55338}")

## As you can see, the MLP has almost **10 times** as many parameters as the CNN.